# Prophet — EMS forecasting
Dùng cùng CSV train/validation/test với LSTM. Chọn kernel `.venv`. Xem `reports/prophet.md` để hiểu protocol dự báo từng giờ với lịch sử đã quan sát.

In [ ]:
from pathlib import Path
import sys
ROOT = Path.cwd().resolve()
if not (ROOT / "ems/models/prophet/train.py").exists():
    ROOT = ROOT.parent
if not (ROOT / "ems/models/prophet/train.py").exists():
    raise RuntimeError("Open notebook from project root or notebooks directory")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
from ems.data import load_data
from ems.models.prophet.train import run_experiment
import pandas as pd


## 1. Kiểm tra dữ liệu và thời gian

In [ ]:
splits = load_data()
pd.DataFrame([{ "split": name, "rows": len(frame), "start": frame.ds.min(), "end": frame.ds.max()} for name, frame in splits.items()])


## 2. Huấn luyện và chọn cấu hình
Fit train; chọn theo MAE validation; đánh giá test sau khi chọn. Cell này chạy bốn cấu hình và ghi artifact.

In [ ]:
metadata = run_experiment()
metadata["selected"]


## 3. So sánh trên cùng timestamp
Dùng `lstm_seq24_aligned` khi so với LSTM seq24. Lag/rolling dùng lịch sử thực tế trước từng giờ; không diễn giải kết quả thành dự báo nhiều ngày từ một cutoff.

In [ ]:
display(pd.read_csv(ROOT / "outputs/prophet/validation_results.csv"))
pd.DataFrame(metadata["test_metrics"]["lstm_seq24_aligned"]).T


## 4. Biểu đồ và checkpoint

In [ ]:
from IPython.display import Image, display
display(Image(filename=str(ROOT / "outputs/prophet/predictions.png")))
display(Image(filename=str(ROOT / "outputs/prophet/components.png")))
from ems.models.prophet.model import load_model, predict
model = load_model(ROOT / "checkpoints/prophet/best_model.json")
predict(model, splits["test"].head(24))
